# COGS515 Lab: Playing Tic-Tac-Toe with Minimax

**Week 4 · about 30 minutes**

> Check Python Bootcamp first if you are not proficient with Python.

**How this lab works**
- ✏️ means answer on the worksheet (pen and paper). 💻 means run a cell.
- **Predict first, then run.** Write your prediction on the worksheet *before* the cell is run.
- **No laptop?** No problem. The instructor runs this notebook on the projector, and every task can be answered on the worksheet.
- **With a laptop:** open this notebook in Google Colab (*File → Upload notebook*) or Jupyter. Nothing to install. Run the cells in order, but wait for the class at each ✏️ task.

| Part | Minutes |
|---|---|
| 0. Setup | 1 |
| 1. Minimax by hand | 8 |
| 2. How big is the tree? | 5 |
| 3. Alpha-beta pruning | 10 |
| 4. Cutting the search short | 5 |

## 0. Setup 💻 (1 min)

Run the next cell. A **state** is a tuple of 9 squares, each `'X'`, `'O'` or `'.'` (empty), numbered like the 8-puzzle positions:

```
0 1 2
3 4 5
6 7 8
```

`TicTacToe` has the methods from the lecture's *Formalizing the Game Setup* slide: `player`, `actions`, `result`, `terminal` and `utility`. X moves first and is **MAX**; O is **MIN**. The utility is +1 if X has won, −1 if O has won and 0 for a draw.

In [ ]:
import math
import random


class TicTacToe:
    LINES = ((0, 1, 2), (3, 4, 5), (6, 7, 8),     # rows
             (0, 3, 6), (1, 4, 7), (2, 5, 8),     # columns
             (0, 4, 8), (2, 4, 6))                # diagonals
    initial = tuple("." * 9)

    def player(self, state):
        """X moves first, so it is X's turn when both have played equally often."""
        return "X" if state.count("X") == state.count("O") else "O"

    def actions(self, state):
        return [square for square in range(9) if state[square] == "."]

    def result(self, state, action):
        s = list(state)
        s[action] = self.player(state)
        return tuple(s)

    def winner(self, state):
        for a, b, c in self.LINES:
            if state[a] != "." and state[a] == state[b] == state[c]:
                return state[a]
        return None

    def terminal(self, state):
        return self.winner(state) is not None or "." not in state

    def utility(self, state):
        """+1 if X has won, -1 if O has won, 0 otherwise."""
        winner = self.winner(state)
        if winner == "X":
            return 1
        if winner == "O":
            return -1
        return 0


def board(rows):
    """board("X.O/O../.X.") -> ('X', '.', 'O', 'O', '.', '.', '.', 'X', '.')"""
    return tuple(rows.replace("/", ""))


def show(state):
    """Print a state as a 3 x 3 board."""
    for row in range(3):
        print(" ".join(state[3 * row: 3 * row + 3]))      # the 3 squares of this row


game = TicTacToe()
Q = board(".OX/OX./OX.")
show(Q)

## 1. Minimax by hand ✏️ (8 min)

```
  Board Q, X to move
       . O X
       O X .
       O X .
```

Board Q has only three empty squares, so its whole game tree fits on the worksheet: 14 nodes. Squares are numbered as in Part 0; an edge labelled `X→5` means X plays square 5.

- ✏️ **Task 1a.** Write the utility (+1, 0 or −1) in the box of every **terminal** board: someone has three in a row, or the board is full.
- ✏️ **Task 1b.** Back the values up the tree. A **MAX** node (X to move) takes the *largest* value of its children; a **MIN** node (O to move) the *smallest*. What is the value of Q, and which move should X play?
- ✏️ **Task 1c.** Moves 5 and 8 both make an X threat in column 2 (squares 2, 5, 8). Use your tree: what does O play next after each of them?

Then run the three cells below to check. The first is the lecture's MAX-VALUE and MIN-VALUE, which you have just traced by hand. `minimax_values` prints the minimax value of each of X's moves: **+1** means X can force a win, **0** a draw with best play on both sides, **−1** that O can force a win.

In [ ]:
visited = 0    # nodes visited by the last search (global: every call below adds 1)


def max_value(game, state):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = -math.inf
    for a in game.actions(state):
        v = max(v, min_value(game, game.result(state, a)))
    return v


def min_value(game, state):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = math.inf
    for a in game.actions(state):
        v = min(v, max_value(game, game.result(state, a)))
    return v


def minimax(game, state):
    """Minimax value of a state with X (MAX) to move; also counts the nodes visited."""
    global visited
    visited = 0
    v = max_value(game, state)
    print(f"value: {v} | nodes visited: {visited:,}")
    return v


def minimax_values(game, state):
    """The minimax value of each of X's moves (X to move)."""
    for a in game.actions(state):
        print(f"X plays {a}: value {min_value(game, game.result(state, a))}")

In [ ]:
minimax_values(game, Q);

In [ ]:
minimax(game, Q);

## 2. How big is the tree? (5 min)

- ✏️ **Task 2a.** Predict: how many nodes will minimax visit from the **empty** board? *Hint:* there are 9! = 362,880 orders in which nine moves can fill the board.

In [ ]:
minimax(game, game.initial);

- ✏️ **Task 2b.** Each leaf of the tree is a finished game. Predict: are there more or fewer possible games than 9! = 362,880? Then run the next cell.

In [ ]:
def count_games(game, state):
    """Number of different games (sequences of moves) that can follow state."""
    if game.terminal(state):
        return 1
    return sum(count_games(game, game.result(state, a)) for a in game.actions(state))


print(f"{count_games(game, game.initial):,} possible games")

- ✏️ **Task 2c.** Compare both results with 9! = 362,880. Why is each one larger or smaller?
- ✏️ **Task 2d.** What value did minimax return for the empty board? What does that tell you about tic-tac-toe?

## 3. Alpha-beta pruning (10 min)

Alpha-beta search returns the same value as minimax but skips branches that cannot change the decision. $\alpha$ is the best value MAX can already guarantee on the path to the current node, $\beta$ the best value MIN can already guarantee.

- ✏️ **Task 3a.** On the worksheet tree for Q, cross out the nodes alpha-beta never visits. It looks at X's moves in the order 0, 5, 8 and at O's moves from left to right. *Hint:* after exploring X→0, X knows it can get at least 0, so $\alpha = 0$.
- ✏️💻 **Task 3b.** In the next cell `ab_min_value` is complete, but `ab_max_value` is missing one line, marked `TASK 3b`: the cut-off on MAX's side. Write it (on the worksheet if you have no laptop). *Hint:* look at the matching line in `ab_min_value`, and at the alpha-beta slide.

In [ ]:
def ab_max_value(game, state, alpha, beta):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = -math.inf
    for a in game.actions(state):
        v = max(v, ab_min_value(game, game.result(state, a), alpha, beta))
        # TASK 3b: if v is at least beta, MIN will never let the game reach this node: stop and return v
        alpha = max(alpha, v)
    return v


def ab_min_value(game, state, alpha, beta):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = math.inf
    for a in game.actions(state):
        v = min(v, ab_max_value(game, game.result(state, a), alpha, beta))
        if v <= alpha:                   # MAX already has a better option elsewhere
            return v
        beta = min(beta, v)
    return v


def alphabeta(game, state):
    """Alpha-beta value of a state with X (MAX) to move; also counts the nodes visited."""
    global visited
    visited = 0
    v = ab_max_value(game, state, -math.inf, math.inf)
    print(f"value: {v} | nodes visited: {visited:,}")
    return v

Now run alpha-beta on Q and compare the number of nodes it visits with your crossed-out tree.

In [ ]:
alphabeta(game, Q);

✏️ Next, alpha-beta on the **empty** board. **Before running**, write your prediction of the number of nodes in the *Alpha-beta* row of the worksheet table (minimax visited 549,946). After running, copy the value and the number of nodes into the same row. The cell also checks your Task 3b line.

In [ ]:
value = alphabeta(game, game.initial)
if value != 0:
    print("Wrong value: alpha-beta must return the same value as minimax (0).")
elif visited == 18_297:
    print("Task 3b: correct. Same value as minimax, 30 times fewer nodes.")
elif visited == 29_431:
    print("The value is right, but MAX never prunes yet: fill in the line marked TASK 3b.")
else:
    print("The value is right, but the count is not what we expected: check the comparison in your line.")

- ✏️ **Task 3c.** `CentreFirst` below looks at the centre first, then the corners, then the edges. Before running: will this change the **value**? Will alpha-beta visit more or fewer nodes than 18,297, and why? Write your node prediction in the last row of the worksheet table, then run the cell and fill in the results.

In [ ]:
class CentreFirst(TicTacToe):
    ORDER = [4, 0, 2, 6, 8, 1, 3, 5, 7]           # centre, then corners, then edges

    def actions(self, state):
        return [square for square in self.ORDER if state[square] == "."]


alphabeta(CentreFirst(), game.initial);

## 4. Cutting the search short (5 min)

Chess programs cannot search to the end of the game. They stop at a depth limit and **evaluate** the position instead. A classic evaluation for tic-tac-toe counts the **lines still open** for each player (a line is open for X if O has no mark in it):

$$\text{EVAL}(s) = \frac{(\text{lines open for X}) - (\text{lines open for O})}{10}$$

Dividing by 10 keeps every evaluation between −1 and +1, so a real win always counts for more.

```
  Board P, X to move
       X . O
       O . .
       . X .
```

- ✏️ **Task 4a.** Compute EVAL(P) by hand. Then run the next cell and compare EVAL(P) with the minimax value of P.

In [ ]:
def evaluate(game, state):
    """EVAL(s); at the end of the game, the true utility."""
    if game.terminal(state):
        return game.utility(state)
    lines = [[state[i] for i in line] for line in game.LINES]    # the 8 lines, as lists of marks
    open_x = sum(1 for line in lines if "O" not in line)
    open_o = sum(1 for line in lines if "X" not in line)
    return (open_x - open_o) / 10


def h_value(game, state, depth):
    """Depth-limited minimax: at the depth limit, EVAL replaces further search."""
    if game.terminal(state) or depth == 0:
        return evaluate(game, state)
    values = [h_value(game, game.result(state, a), depth - 1) for a in game.actions(state)]
    return max(values) if game.player(state) == "X" else min(values)


def play(game, depth):
    """One game: depth-limited X against an O that moves at random. Returns X's utility."""
    state = game.initial
    while not game.terminal(state):
        if game.player(state) == "X":                # X picks the action whose result has the highest value
            action = max(game.actions(state), key=lambda a: h_value(game, game.result(state, a), depth - 1))
        else:
            action = random.choice(game.actions(state))
        state = game.result(state, action)
    return game.utility(state)


P = board("X.O/O../.X.")
print("EVAL(P) =", evaluate(game, P))
print("minimax value of P =", max_value(game, P))

The next cell plays 100 games of a depth-limited X against an O that plays at random. A depth-1 player looks only at its own next move; a depth-2 player also looks at O's reply.

In [ ]:
for depth in (1, 2):
    random.seed(0)                                   # fixed seed: the same results on every run
    results = [play(game, depth) for _ in range(100)]
    print(f"depth {depth}: X won {results.count(1)}, drew {results.count(0)}, lost {results.count(-1)}")

- ✏️ **Task 4b.** Why does the depth-1 player sometimes lose, even to a player that moves at random?

✏️ **Discussion** (in your group, then with the class)
1. Rules such as "win if you can, block if you must, take the centre" are a kind of evaluation function. What do they know that EVAL does not?
2. Minimax needed 549,946 nodes to play perfectly. How many positions do *you* think you look at before a move?

